# Notebook 3 — Baseline ML Models (R3)
Rice Variety Classification | F20DL 2026/2027

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay

df = pd.read_csv('../data/tabular/rice_cammeo_osmancik.csv')
feature_names = [c for c in df.columns if c != 'Class']
X = df[feature_names].values
le = LabelEncoder()
y = le.fit_transform(df['Class'])
CLASS_NAMES = list(le.classes_)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

results = {}  # collects metrics for comparison table

## 1. Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
results['Decision Tree'] = {
    'Accuracy': accuracy_score(y_test, y_pred_dt),
    'Precision': precision_score(y_test, y_pred_dt, average='weighted'),
    'Recall': recall_score(y_test, y_pred_dt, average='weighted'),
    'F1': f1_score(y_test, y_pred_dt, average='weighted'),
}
print(classification_report(y_test, y_pred_dt, target_names=CLASS_NAMES))

plt.figure(figsize=(20, 8))
plot_tree(dt, filled=True, feature_names=feature_names, class_names=CLASS_NAMES, fontsize=9)
plt.tight_layout()
plt.savefig('../results/decision_tree.png', dpi=100, bbox_inches='tight')
plt.show()

print("Discussion: The decision tree uses morphological features like Extent and Eccentricity as "
      "primary splits, reflecting that Osmancik grains are more elongated than Cammeo.")

## 2. Naive Bayes

In [ ]:
nb = GaussianNB()
nb.fit(X_train_sc, y_train)
y_pred_nb = nb.predict(X_test_sc)
results['Naive Bayes'] = {
    'Accuracy': accuracy_score(y_test, y_pred_nb),
    'Precision': precision_score(y_test, y_pred_nb, average='weighted'),
    'Recall': recall_score(y_test, y_pred_nb, average='weighted'),
    'F1': f1_score(y_test, y_pred_nb, average='weighted'),
}
print(classification_report(y_test, y_pred_nb, target_names=CLASS_NAMES))

## 3. Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
results['Logistic Regression'] = {
    'Accuracy': accuracy_score(y_test, y_pred_lr),
    'Precision': precision_score(y_test, y_pred_lr, average='weighted'),
    'Recall': recall_score(y_test, y_pred_lr, average='weighted'),
    'F1': f1_score(y_test, y_pred_lr, average='weighted'),
}
print(classification_report(y_test, y_pred_lr, target_names=CLASS_NAMES))

## 4. Perceptron

In [ ]:
perc = Perceptron(random_state=42, max_iter=1000)
perc.fit(X_train_sc, y_train)
y_pred_perc = perc.predict(X_test_sc)
results['Perceptron'] = {
    'Accuracy': accuracy_score(y_test, y_pred_perc),
    'Precision': precision_score(y_test, y_pred_perc, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, y_pred_perc, average='weighted', zero_division=0),
    'F1': f1_score(y_test, y_pred_perc, average='weighted', zero_division=0),
}
print(classification_report(y_test, y_pred_perc, target_names=CLASS_NAMES, zero_division=0))

## 5. k-Nearest Neighbours

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_sc, y_train)
y_pred_knn = knn.predict(X_test_sc)
results['kNN (k=5)'] = {
    'Accuracy': accuracy_score(y_test, y_pred_knn),
    'Precision': precision_score(y_test, y_pred_knn, average='weighted'),
    'Recall': recall_score(y_test, y_pred_knn, average='weighted'),
    'F1': f1_score(y_test, y_pred_knn, average='weighted'),
}
print(classification_report(y_test, y_pred_knn, target_names=CLASS_NAMES))

## 6. Ensemble — Random Forest & Gradient Boosting

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results['Random Forest'] = {
    'Accuracy': accuracy_score(y_test, y_pred_rf),
    'Precision': precision_score(y_test, y_pred_rf, average='weighted'),
    'Recall': recall_score(y_test, y_pred_rf, average='weighted'),
    'F1': f1_score(y_test, y_pred_rf, average='weighted'),
}
print('Random Forest:')
print(classification_report(y_test, y_pred_rf, target_names=CLASS_NAMES))

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
results['Gradient Boosting'] = {
    'Accuracy': accuracy_score(y_test, y_pred_gb),
    'Precision': precision_score(y_test, y_pred_gb, average='weighted'),
    'Recall': recall_score(y_test, y_pred_gb, average='weighted'),
    'F1': f1_score(y_test, y_pred_gb, average='weighted'),
}
print('Gradient Boosting:')
print(classification_report(y_test, y_pred_gb, target_names=CLASS_NAMES))

## 7. Model Comparison Summary

In [ ]:
summary = pd.DataFrame(results).T.round(4)
summary = summary.sort_values('F1', ascending=False)
print(summary.to_string())
summary.to_csv('../results/baseline_comparison.csv')

ax = summary[['Accuracy', 'Precision', 'Recall', 'F1']].plot(
    kind='bar', figsize=(12, 5), ylim=(0.8, 1.0), rot=30, colormap='tab10'
)
ax.set_title('Baseline Model Comparison')
ax.set_ylabel('Score')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../results/baseline_comparison.png', dpi=100)
plt.show()

In [ ]:
# Confusion matrices for top 3 models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, preds) in zip(axes, [
    ('Random Forest', y_pred_rf),
    ('Gradient Boosting', y_pred_gb),
    ('Logistic Regression', y_pred_lr),
]):
    ConfusionMatrixDisplay.from_predictions(y_test, preds, display_labels=CLASS_NAMES, ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.savefig('../results/confusion_matrices.png', dpi=100)
plt.show()